In [ ]:
%pip install ipywidgets mrcfile
import io, warnings
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from scipy.ndimage import rotate as nd_rotate, zoom as nd_zoom, gaussian_filter
from scipy.signal import convolve2d
from ipywidgets import IntSlider, FloatSlider, Dropdown, VBox, HBox, Layout, Output
from IPython.display import display, Image, clear_output
warnings.filterwarnings('ignore')

slider_layout = Layout(width="420px")
label_style = {"description_width": "150px"}


def figure_to_image(fig, dpi=100):
    """Render a matplotlib figure to a PNG and return it as an IPython Image for notebook display."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    return Image(data=buf.read())


def make_letter_image(text, size=64):
    """Render a single text character as a binary 2D image of the given pixel size.

    Steps:
      1. Draw the character using matplotlib on a small canvas.
      2. Extract the rendered pixel buffer as a grayscale (averaged RGB) array.
      3. Normalise to [0, 1], then invert so the letter pixels = 1, background = 0.
      4. Flip vertically so the image matches the standard mathematical orientation.
      5. Zoom (resize) to the requested pixel size.
    """
    fig, ax = plt.subplots(figsize=(2, 2))
    ax.axis('off')
    fig.text(0.23, 0.26, text, fontsize=100)
    fig.canvas.draw()
    raw_buffer = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    canvas_width, canvas_height = fig.canvas.get_width_height()
    # Average the R, G, B channels to get a grayscale image
    grayscale = raw_buffer.reshape(canvas_height, canvas_width, 4)[:, :, :3].mean(2)
    plt.close(fig)
    # Normalise pixel values to [0, 1]
    grayscale = (grayscale - grayscale.min()) / (grayscale.max() - grayscale.min() + 1e-10)
    # Invert (letter becomes bright) and flip vertically for correct orientation
    binary_image = (~grayscale.astype(bool))[::-1].astype(float)
    return nd_zoom(binary_image, size / binary_image.shape[0])


def power_spectrum_2d(image):
    """Compute the log-scaled 2D power spectrum of an image.

    The 2D FFT is computed and shifted so the zero-frequency component
    is at the centre of the array. The power spectrum is log(|FT|^2 + 1);
    the +1 prevents log(0) at pixels with zero Fourier amplitude.
    """
    fourier_transform = np.fft.fftshift(np.fft.fft2(image))
    return np.log(np.abs(fourier_transform) ** 2 + 1)


# ---------------------------------------------------------------------------
# Image kernels used for convolution demonstrations
# ---------------------------------------------------------------------------

def sobel_kernel_x():
    """Horizontal Sobel kernel — detects vertical edges by computing the image gradient in x."""
    return np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=float)


def sobel_kernel_y():
    """Vertical Sobel kernel — detects horizontal edges by computing the image gradient in y."""
    return np.array([[ 1,  2,  1],
                     [ 0,  0,  0],
                     [-1, -2, -1]], dtype=float)


def laplacian_kernel():
    """Laplacian kernel — approximates the second spatial derivative; highlights regions of
    rapid intensity change (edges and fine features)."""
    return np.array([[0,  1, 0],
                     [1, -4, 1],
                     [0,  1, 0]], dtype=float)


def gaussian_kernel(size=9, sigma=2.0):
    """Build a 2-D Gaussian blur kernel of the given size and standard deviation.

    The kernel is the outer product of two 1-D Gaussians, then normalised so
    all weights sum to 1 — this preserves the total image brightness.
    """
    axis = np.arange(-(size // 2), size // 2 + 1)
    gaussian_1d = np.exp(-axis ** 2 / (2 * sigma ** 2))
    kernel_2d = np.outer(gaussian_1d, gaussian_1d)
    return kernel_2d / kernel_2d.sum()


def box_blur_kernel(size=5):
    """Build a uniform (box) blur kernel — every pixel in the neighbourhood is weighted equally."""
    return np.ones((size, size)) / size ** 2


def sharpen_kernel():
    """Sharpening kernel — enhances fine details by subtracting a blurred version of the image
    from itself (centre weight = 5, neighbours = −1 → boosts the Laplacian residual)."""
    return np.array([[ 0, -1,  0],
                     [-1,  5, -1],
                     [ 0, -1,  0]], dtype=float)


# ---------------------------------------------------------------------------
# Test-pattern generators
# ---------------------------------------------------------------------------

def make_ring_pattern(n=128, radius=0.3, width=0.04):
    """Create a binary annulus (ring) image."""
    y, x = np.mgrid[0:n, 0:n].astype(float) / n - 0.5
    radial_distance = np.sqrt(x ** 2 + y ** 2)
    return ((radial_distance > radius - width / 2) &
            (radial_distance < radius + width / 2)).astype(float)


def make_horizontal_line(n=128, position=0.0, width=0.03):
    """Create a horizontal line pattern at a given fractional vertical position."""
    y_coords = np.linspace(-0.5, 0.5, n)
    return (np.abs(y_coords - position) < width).astype(float)[:, None] * np.ones((n, n))


def make_dot_pattern(n=128, radius=0.06):
    """Create a small filled disc (dot) image."""
    y, x = np.mgrid[0:n, 0:n].astype(float) / n - 0.5
    return (np.sqrt(x ** 2 + y ** 2) < radius).astype(float)


def relativistic_electron_wavelength(voltage_eV):
    """Compute the relativistic de Broglie wavelength of an electron in Ångströms.

    Formula: λ = 12.264 / sqrt(V * (1 + V * 0.98e-6))  [Å], where V is in eV.
    """
    return 12.264 / np.sqrt(voltage_eV * (1 + voltage_eV * 0.98e-6))


def make_frequency_mask(n, cutoff_low, cutoff_high=None, filter_type='Low-pass'):
    """Build a binary mask in Fourier space for frequency filtering."""
    y, x = np.mgrid[0:n, 0:n].astype(float)
    y -= n // 2
    x -= n // 2
    radial_frequency = np.sqrt(x ** 2 + y ** 2) / n
    if filter_type == 'Low-pass':
        return (radial_frequency <= cutoff_low).astype(float)
    elif filter_type == 'High-pass':
        return (radial_frequency >= cutoff_low).astype(float)
    else:  # Band-pass
        return ((radial_frequency >= cutoff_low) &
                (radial_frequency <= (cutoff_high or cutoff_low))).astype(float)


def make_box_signal(box_width=200, num_samples=1000):
    """Create a 1-D box (rectangular) function."""
    signal = np.zeros(num_samples)
    centre = num_samples // 2
    half_width = box_width // 2
    signal[centre - half_width : centre + half_width + 1] = 1
    return signal


def fourier_series_reconstruct(signal, num_harmonics):
    """Reconstruct a signal from its Fourier series up to num_harmonics terms."""
    num_samples = len(signal)
    sample_indices = np.arange(num_samples)
    reconstruction = np.zeros(num_samples)
    for k in range(num_harmonics):
        cosine_coeff = 2 / num_samples * np.sum(signal * np.cos(2 * np.pi * k * sample_indices / num_samples))
        sine_coeff   = 2 / num_samples * np.sum(signal * np.sin(2 * np.pi * k * sample_indices / num_samples))
        amplitude = np.sqrt(cosine_coeff ** 2 + sine_coeff ** 2)
        phase     = np.arctan2(sine_coeff, cosine_coeff)
        reconstruction += amplitude * np.cos(2 * np.pi * k * sample_indices / num_samples - phase)
    reconstruction -= reconstruction.mean() - signal.mean()
    return reconstruction


# Pre-computed test images shared by multiple widget functions
test_image_64px          = make_letter_image('Q', 64)
test_image_128px         = make_letter_image('Q', 128)
test_image_128px_fourier = np.fft.fftshift(np.fft.fft2(test_image_128px))

# TUD flame — embedded as base64-compressed uint8 (128×128) so it loads in any
# environment including JupyterLite, which cannot access the local filesystem.
import base64 as _b64mod, zlib as _zlib
_FLAME_B64 = 'eNrFW2uQbFV13jPd57X3eXX3XC+WT0CvloHSqCABqaiIiGgFq0JIWSrGZ8VoGRJ/xFdU8rDQVIwxVsWKmoqWRqWImmAUTRFIKkGUxCBGFFB5CN470336vB89j51v7XP6NdPzuiCZrjs9c3u69/rW+ta31tpnHyH2/OLzP3N8mRb9aNvC4ty2OVe/qe/qjzlvnsbf1PsmL+//xfme1tgOrcm5hcXV38IK9fk2H791vBSfmDGxjx/QiEX4xQx+e3f84mHFz3fidwhrjd+ipefw8180fuv/G79L+C1rDr+1D37xMOI3TIXfEcqERxS/mMPvEH57Fr/YF/8BjLDnl+ez7yH8OuF3wEKTTOBY2lL4rUcEv+14LrCapqnwW48sfgH6awbc7zhujZ8ssBQJ5/GLhxM/n4HvdjxgNQyDuxSCRwx//UbL1JfblrA8zxOGCRMefvwW37b87Nsdr/OonsNtTdO459qG6cACpCJcsBj/dgMOgn+nAc1PXOgMX23L6va6tqZbnmOatg0qIhGVAb8A/HzyDNo/7tJXvuriZ526AivMLkwwXNcywQPgdx4Sfj552DYXcw8gw8MxXb3DLpWyGCYP3PzF915yKmP6imdqrgMegAI1B+15k08Cv2lNlt5mGoywXvXtYmMQSSlHd33ht2CC1zUMVxiggGOJObvFzAccJv6uPbP8XObg4Syf8rpvVls/H64GpZQ//vjzdOb74AEoYKq+ZG7xk4q/MYefz5HAxkpMe/HXZTYqo2CtlOk/vkxnXUeDGrl74RcHxs99dyH++tk27SMdZlwZh2FejKJ+KKvrLmB6D2pkOHw+/ieJf1mfD8AcACRZS/M9dumJKowHeVkMhjL75DHmO7pn1hZux8/3xE8WjwlOFPf1v7yY9TTPcE3HUjldv6w+mQho+55h+OzMO6qkisMIFoTy3jfreldz+SL+b0vhfR62fq78GOtgdVrTrlWFTwMBodMNxzVW9Cf9TznI85ws6Ofy2mOsa0A76nftEv9FCT+X5Q6A/Ym8xtlu7VSIwXHPgeK0HP3U729EcVTAgirpy3suY77Cb+0a/4V6s/3/vG/Km7zty892VzbywzW7lqc99e48z2BBFiZVv6j+VBf2Dvx8O36+QPjGHHBMr3XGfeu3dMYRn6099a/QWc+G3LQghksvjIM8KYdplgVVNJRfepThWlP8fAb/7vGf4Z7htVfYrxfl7Y6nu8S/5tH8SWOTo8MEo2f4Wo+9TQ6KoIrTqAryfLD1X8fa4A0fD2FciL24Z8/gVqvrfusI+yMZfW+8/hhM01rUP1mujXKj2XhHV/uCHIzggzwoo7gYVvec3fLMphGYtWCPtefwA9K1GwXWr9Fb9cNuEl+oJtNGfXAEUcQS2pkPpnGWxWkepXlYhvnq85f8xm57u/7vtGVClaktvduL/LZOvXbDgWn+49mFyOCfaZG5Ro+9AxEYllGSZmE5LOI0fDHz9frtosmF6SqLHnNUcNpPD8LN249a2+p/7fvaBHScKNFe7Qzx6NvLGOCzOMmjPE7TKHxR4wH7YIIzI3+u3mWXrg9Gd57ebsSvCYByf51YKE8uUtBuOxb+vN1lb5erVUguSNMkTeMiCs9vIQuEzQ8mfjPh90H/t8tBdeIs1tEbAR4HQH0IntHvYeLAEEDg0X+bT7gL/E8APYvBgxQ/H38mjCOh3GvlGfLP0f9jMoyLy9hK29dds0lBa2oDGj30exgCdIvKrat12Ufk6vowj+CCYRHmSRYXPzodpdBu8n9X4u0gv+63e+zLm3Ek/5AdaU090NCv7q5Mx8Pk7zodDGKu5pod9gKsGhVwP/BnSZJk0ehbXZTphoH2rBmL4j7jAa2zfNN6Em7+03K3LoCm8oGywamj4VhtRziGuYwBWPdNqIB/y3qUR1kKGuIJNCwC+UUNFQxi6DRMnGKd+3mOpRRc17x1MyiSB55kzlCfKxRqh8NyNL8rdNPu9hyNMsEDZT8oT4zg/CqohlAApOGoL68iBikfLEiDXadL/HabXFs/Id/AunoNnI+3VmoJtDotduTU07xlhjap3TUcrH9JRYGPkywhAiILonw4egXrtr2GPg3X9jSkKUTiB1v99b78BnesKUdEYwOWd9l5H//2PT/978+88bEM8muD5Mbj7i4QeMV/in+ibHjgTCw/aR/20p+Z/Dc8884qjrOweCn8Ny5AYyfg71ztHYncKsqRlHde9TjmQwz0buvL0MCIOEgCRDxEMmze6DSMmYW+Y9aYFMBmfeMHI4j5cOtrOsWveX9DIwfjx7vkYLiWhsmJYS7veC3zSITYe5CBAUKP9XOEogyrYLQqr2boyMwZ3m2LQBOVGYuILt/ZGuDd/a1XNQxQPb9KAsfosAvzIK/KLM2rKlor5KeOtKkKv3w9IO7nJEAZSVCcx2DEJfQR4xJm7VIFppkIkMjAG+SgjIqwuuPxppqpx07EaxCEr2+FRT8JsygcZqO4L28+3UA/+NSfQ/8QeiXAaZIpAoSj7z+6GUo5X8D7Gf2d5n+XXSMDUDkfys+wXruj0cMz1OjX6rELkjhJyhwLlHkSZGWwcceTMXl2vrMxpNAnac3/mMpA1Zd/hY/wlRcbFyx6zHofHv6QPL6JDNg4Id/CfEOhB+tBDFfvsffKtdEwGWKBME6LFHHavO2J7c7y5/D/QRWUyP+MSBDnYQ5FKF9MLN5HASYKWLcfvyP7YBIVkvjX2Eqr2+q2fYW/u7zC/o6IXqVwdpbHUUKas/5N7Sh6prUqVOGPC8gg2VDExWDz5s44rRc2gGNOTioRuu8XlgHKKT2i/tl1GRYmhZ6I/nmQI44jLJSEeVr019c2T8h3tluvlCfWQdsyRBWO6/BH6AkH8l0qAp4+7qbms6Ep6DOssI0n3Z8rIlEdu/dZS90W4g8mIveA/y9kvxpWSQG1z+M4oYRPsuPnsktR+iD9KcQHBRBLowygFUmzB56mT5i/226LNScA5g2b9FkxiWl+31lLSCEUOhqHUOpeT/HPUGzzYRaj6cLflMHWNewiqn2UgkWc4J3KBnxKtSY/iVGO3msvgG7PdaFjAlxNsSQLgCY7fgFy2BEmeg5X97Sn/CyHzubU5iQwg2rdYH1VXvpS+B8dcB6rF4A/TfM8oZYsegFmuR0CsLAlsOoEeAnFkeRUPYZXsJ4BH3FX66DZ+iQcMIDGRSniT1pXQGqrB+4oqQBQ6icReiHqhsIkxscM5LUaMtBYKECLx8FTflCpUNIjycKN9yzbrubDBm757HxwL4UPmvAmJDr4Q+rA0rr+0QsJ+FFkKTQBw8lL4EHP3EUBJ7nPpx3oxyjIjQvCck3+7crySsuHEDvaCvsEtds5dbzwNH2RqZlquzLkXRHFZGCaRfEQDoImyOuoDzC3yf8s5jkKoMS8qBgWSsuJAshiecvTWdcyfVIS49j9GSQ+IY0l/GldbfMaPwpnlmcJEjTJcwxEsapJF7OOOW6nF9Bv3v20sfxvW00xU/0s4nvicmZ7LRAZ9HizXFPJFsIOEppEETWpkw+hDyDQWRknQRyP8Feog19ECru7B2DnBPYmJYGR6ieorQ2T9T9ztF7bpf7D/LJEnwV1q7VOcS5VZYdKf5iN0iQIgzQflWmQZSmKcXTe2AF8Ue7NU5EK4tHbR2HtfyQ1lDBPA3nD01jPxNjrtc+4X5U5wkq9jqIqqYViwGjYj6qNrVERDFaLIqdSOpAfZcoB4xBsc8L89o9ywNvGDkBoycVx2d/42W8wx213MKG8CSI8xNAV0qJpLVRx89zfkMM7vvUf3723kDJZS6qEmqJ7TqOt2d2mkG3xJ2O6t66TAyi11GdjtAzT0dW21oUB3eW/x4g0RGTiOuMV/rrsRvKWt5191BdHTnvR71+3KtMgHq2hm3wrVQFj0ovPO2B+HKhT8BUbA8XAuqmDq1UMvnbaErSMm8fuBVb0+5QETa1Q35GW73HZMrddrjNmPOMDP5FhmGcDeb29QwCmqBdMoZ3WNXLQ9HPQIiwPwcuDjbtfsOQKlIHXUgRIfbO4NqF2/rB6M+t0OqbQbcwHXoud/qFga5DlYXzW0mIDZvFP/wdDxhkPpo0CUq5HNNcpJ7xmGcWgo1MOQIFy0r86ShkR7ePslLbV9qyOYbcMreNz9tyvopoM5PtoQ3EX/tnbBrEmB7cGqp6o6OZqukAoos33L0OHlp4ToMYUU9ertit78Jd0YXWFbXLuOB3B2263Zbw7QoG63nLNaRGaf8wrYG0Aiv2nqdVRFKQUICHKVSw/0e5Chj8MCgZ5XXRSqgNUaj6/hDxvYUggEzQhOqbuQk7v3Oqnz2q7++1IbCtER1QOKITkfpUNabhxXH4GLxpPuR/2qOjUea/azbeyI8sd7usOst0XrsHNXpt77SfcmldXshXNW8z/nUOJGkTbz/w58aoGTp4grGV/67j8lBAe2tTValDRjhOVYNRptG0XM9/EVModqte6zT3T4p2lI0uPuUZ+ZG5HebcNoZntOHSCl+RDpUI0TTRsjKsBZtsPs0exZx9XZShJ68wnL0TPWQI3hGMKJKlLwzFZgOJnW39w7rIaBncbQLZvChAJOuyKjT4pPQ10qQq1kvMi2LqSncK+IgegP0xSuk9K1H8mBdkQQh3LwAAPqbBdVE3usrbNhVjcBC/Gr2ToLVv9EvihQorfCW1z0nSfnsPZFbTxiP5Haa/iYHgOzfwc3R7aHcLPuWfRLo1tHHG5WNT07MQ/U4dNn70V/S7ta5ECgQa0x4WfhvLGrn7GGn6pQ5Ao/gcbl1OvqSGFaGPOsDkaV0GbINxpT/ZExX6bAJOdADUNvGkjyMnvETWkaHzgAgwd6PlZ73q5OoIMVmBeGYwCDEwfpE6LLkMh/g7FgLsWbQKhc7b5zusRYreIzHajvxlTzDO0GuQFJToke8d/lX2aFKqoe33kaBauf+8o3oiRU9Dum0Vb5aatdg8cfeba7N7nK2aNUh54/v0IcxKV1NdCgBOKd1ik37fejyKNKYCaUdr/i8rB5hsQgA6aPWKSavptQYMDuhan2UBb7P5dNkVVKVp66n/Kfo5iT5usKeo+inFU9OW3vr1R44/rhhBSMbr9qLA4PE5rW3QZ3aBL44SDz18aFPtegx3LEO0KfkLGYRlH8H5ShOSGNEFTuA7Kx+N5K0erWa1CGVYMHxz0TJc4aIH/Nl2y8ybng/Zh3o4cQPycpd/ub/bzIkyKBJ5PImrxiziMm80OmjfIhKgaji5nvtYxOrpj0oYZiDjGLzjfZ/5YYEJdLs0uO+smGQ3LdJjCiDSPErXdnxF+ZUJCMaCRN1k9p63wW/UZMfCfNlFM157gF/teAt2mhcLRey3n/eHmIK7iYQGdRYuPfzEmjcYEJYP0HBd3HUOn5WPoJRVA/NU1aXjDPsT15/lShChqrsuee4Ms1rIqjPMcM1iuFh+bMHVDNPrhsZYH/Eg8uNzlJh0KgDeIBwe9/jovyIiiZ5q9ZXHlPTLul2hCijyOyIJm8TH+jHYgh8WPnr7cQxE0SXKN2gzHbDYxD4+frHfxMZrnsmMfDeUgKIowyguyIJ/Bn2cKPyha3vc8jGs23jSPnx8cv5ifD2j6hwtWdHbetaOtICjLKMyhSTCBQkAbAglGPjIFKQqBejVDB+zTlUrbbfA7J4dfXeRBDiKTnLbTXdJedv36VjAowMSUwkBewLxZ448pFxGE4eiPTd5pE3M4SgH037HNh4Dfot0PDMAWXXc3L/uXkRz28yoNw0yZkOZT/AntyZRD+Q+PZysmdN8m+eWOdUj8Yq4/oeNFkCHD9gyj4zH+8q8WMsJwVcZhBBMQiCRr8Cs/hJDnOy9irm34luFO8dsn5QG1hQr7LRfOdAyj6zLjos8NZdEfZFURD8O0gCiEZAOGBPJDGVVrRfJOy/DbMJkO5hwev9iG37LJBTDC1c2ux5bP/vO7acIMsqpMwmEMN6QUiqwg/Bj7k0h+5RhbsTRPERhC/pDwc1CA2jnbND1HNzqdZfb4N17Xl1vx2iCtqhxuiPOiSJQfyiQPknwg772cia6GLsA11WVTbj8E/IiioLNmjkE+0L2eyfRffseNQymz/lqYj8osQiiyElVymMYVpHgt2/zrI6yH9ovidlj8M9efVA0h/OraK7hkuq6piZ7fYuLst//zg1KOorV+VIzKNArCJC/TLCApTvryuxcxrA3x8EwxmX8OjZ+LxgV02hNa4pim4wnNcFfcZWY87dV/c1siZTlcG8RkQxhgTkBbFCSjYBS9z9E6moM2fHpA77D4VRFQLqArn3Ti0XEsQ3iu2Ta9nq8xduT83/vCDwsp8+HqICnXs3QQpmmVDlCq5TeewbqmThuIh8MvZvGLGj+dN27OXDrCMG3fMTXD6XUFY0uPufBdX/lJBToEq2GyXiYx8KMgrMm1NzK70/K41TSBh8cvJvhFzQH4wRRoKg3L8RxL07i/0rEYaz/xZVd9/WcbciNZHaT5ejaAGI76sfz0UXZEQ+Qc66Tx8wZ/zQGLTkCgqLo2TLA9zzY0XXRWfJMx49jlH745UG6I1ykZRvmq/N8LWO/Q+MUO/PSb1Zy5VUd/lQncgNJ7rjDaBm276Iw55/zul+7flMXqMC3LQbiOOe1KZqtd6MNnwOQAxDx+ZQaYABMcR9CZbNdD1dVM0MFFKJ78us/+VG4l/WBU9qGNG5/qar6hrojv34bvwC+m+PkYvyD8nG5GMPFER3GVDaCDbvpdX2fLp7/uq325sTZYT6I0lP/+lCWfqrElDjIJ7DCgPmRen3ZXz3RMlE5fwxDgp6PIygbQwXV0XbdXYEL72VfdJuVqVKRpsHnXrzB/5ybA3lbMnH2awd9YQCSwlQn0RDYIR9EB+ujytuFQmTjltTfK9QBda1CduHCpORy0Y//hYPjFdvyUj+pOFDBR2WDCBpf00RS+a6FMrFjMueymzSqIq2EWXrSEyeBgY8j84esZ/KK556BenNaexa+SAvrocN20Xc9si67D+Gu+J9PBKEoH57ZoIN5xsG7eEL6Y/3x8pKy+52AeP2WEOg6vbKAnj4jp+LADPdORD/TlahUVPz61GQRmDhzud0BoevhoAf76O90NwjntO9oTOmDmIy8gO2GBw573rzAg2LpO83R3eimyzqs9D0jNnh/cBb9Q+AXtu6izQcokVx0kxgRsQh7No2zlahmUwdZb2Ep7fLpscv35QPGf4G/+TfBz3sRg/GTRzTE0ATsWmhabLGh3bHbFoAw2bjvFXHDoca99oe3xFw1+3iwupjFoLLKtWp4w/UAShHB14RnmCrvgx1sn5GvGR1tmdoMPFn8xjf8C/HzCA5UVwE+BQvtvk0q2fW74rTPvieRnaZfKmF6NWZCHfJEI7IFfKPw1Dyb4lSW2ujMCQ6hrWX6rx5593/qdj50/Jr3Xjuy2E/g74s/VIUPerF2HgvZdHOHQ4tw1MTnRAE17ka0VdpS9fvTA+eO94J2n9A6U/wvxN2vXptj05NAtcmSHqpkunU93LL5yqzxv2Z0tA/syYHH8xXz8VZ9MBx3ssTtMdexBHUFCBPT6EEGPvftGfy799t8S3AU/3wc/vWbVdwhSz2jx5hARJrKD67+Yzf/GAj7Fz6f8n7PBbu7ioDZB9Yz1/RmElgrmQYrODgN2wy924p+8ZjWusKb3p1jWIW+Am11+T/wzsZjF3/QMdn2CVCnk4W65Oyx+McsNu/nmWKI5CnlS+MWB8O+4/3WcqPb4/iS+7e6qk8Avavy0qn1w/PYEP3/Y8auY7I+f/0Lxi33w10J3Uvjn73+ue40xflscBL+YwX8S9z9P75+YxW/tjV+M22W+7f7fhxk/3wP/9P6HBr94OPA7i/HzRfgf8v3Pe+Gfvf9ZPGL3v+/AP77TpTnpeUj8/wefyf8H'
tud_flame_image = np.frombuffer(
    _zlib.decompress(_b64mod.b64decode(_FLAME_B64)), dtype=np.uint8
).reshape(128, 128).astype(float) / 255.0
del _b64mod, _zlib, _FLAME_B64


# ===========================================================================
# Widget factories
# ===========================================================================

def wave_explorer():
    """Explore how amplitude, frequency, and phase independently shape a cosine wave."""
    amplitude_slider = FloatSlider(value=2.0, min=0.0, max=4.0, step=0.5,
                                   description="Amplitude A", style=label_style, layout=slider_layout)
    frequency_slider = IntSlider(value=4, min=1, max=12, step=1,
                                 description="Frequency f",  style=label_style, layout=slider_layout)
    phase_slider     = FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=0.25,
                                   description="Phase φ (rad)", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        time = np.linspace(0, 1, 1000)
        wave = amplitude_slider.value * np.cos(
            2 * np.pi * frequency_slider.value * time - phase_slider.value
        )
        fig, ax = plt.subplots(figsize=(10, 3.5))
        ax.plot(time, wave, color='steelblue', lw=2)
        ax.axhline(0, color='k', lw=0.8, ls='--')
        ax.set_xlim(0, 1)
        ax.set_ylim(-4.5, 4.5)
        ax.set_xlabel("t")
        ax.set_ylabel("f(t)")
        ax.grid(alpha=0.3)
        ax.set_title(
            f"f(t) = {amplitude_slider.value:.1f} · cos(2π · {frequency_slider.value} · t − {phase_slider.value:.2f})"
        )
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [amplitude_slider, frequency_slider, phase_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([amplitude_slider, frequency_slider, phase_slider, output_area]))
    update_plot()


def fourier_series():
    """Reconstruct a box function from a finite number of Fourier harmonics and watch Gibbs ringing."""
    box_width_slider    = IntSlider(value=200, min=50,  max=500, step=50,
                                    description="Box width (px)", style=label_style, layout=slider_layout)
    num_harmonics_slider = IntSlider(value=15,  min=1,   max=80,  step=1,
                                     description="# waves F",     style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        original_signal = make_box_signal(box_width_slider.value)
        reconstructed   = fourier_series_reconstruct(original_signal, num_harmonics_slider.value)
        x_axis = np.linspace(-500, 500, len(original_signal))

        fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
        axes[0].plot(x_axis, original_signal, color='gray', lw=1.2, label='Box')
        axes[0].plot(x_axis, reconstructed, color='steelblue', lw=1.6,
                     label=f'Reconstruction (F={num_harmonics_slider.value})')
        axes[0].set_xlabel("x")
        axes[0].set_ylabel("Amplitude")
        axes[0].set_ylim(-0.4, 1.5)
        axes[0].legend(fontsize=9)
        axes[0].grid(alpha=0.3)

        num_samples    = len(original_signal)
        sample_indices = np.arange(num_samples)
        harmonic_indices = np.arange(1, num_harmonics_slider.value + 1)
        amplitudes = [
            np.sqrt(
                (2 / num_samples * np.sum(original_signal * np.cos(2 * np.pi * k * sample_indices / num_samples))) ** 2
                + (2 / num_samples * np.sum(original_signal * np.sin(2 * np.pi * k * sample_indices / num_samples))) ** 2
            )
            for k in harmonic_indices
        ]
        axes[1].bar(harmonic_indices, amplitudes, color='steelblue', edgecolor='k', lw=0.4)
        axes[1].set_xlabel("Frequency k")
        axes[1].set_ylabel("Amplitude")
        axes[1].grid(alpha=0.3, axis='y')
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [box_width_slider, num_harmonics_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([box_width_slider, num_harmonics_slider, output_area]))
    update_plot()


def spectrum_explorer():
    """Build a signal from up to three cosine components and observe its frequency spectrum."""
    amplitude_1_slider = FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5,
                                     description="Amp₁",      style=label_style, layout=slider_layout)
    frequency_1_slider = IntSlider(  value=10,  min=1,   max=100, step=1,
                                     description="Freq₁",     style=label_style, layout=slider_layout)
    amplitude_2_slider = FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5,
                                     description="Amp₂",      style=label_style, layout=slider_layout)
    frequency_2_slider = IntSlider(  value=20,  min=1,   max=100, step=1,
                                     description="Freq₂",     style=label_style, layout=slider_layout)
    amplitude_3_slider = FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5,
                                     description="Amp₃",      style=label_style, layout=slider_layout)
    frequency_3_slider = IntSlider(  value=40,  min=1,   max=100, step=1,
                                     description="Freq₃",     style=label_style, layout=slider_layout)
    dc_offset_slider   = FloatSlider(value=5.0, min=-5.0, max=10.0, step=0.5,
                                     description="DC offset", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        num_samples = 1000
        time = np.linspace(0, 1, num_samples)
        signal = (dc_offset_slider.value
                  + amplitude_1_slider.value * np.cos(2 * np.pi * frequency_1_slider.value * time)
                  + amplitude_2_slider.value * np.cos(2 * np.pi * frequency_2_slider.value * time)
                  + amplitude_3_slider.value * np.cos(2 * np.pi * frequency_3_slider.value * time))
        fourier_coefficients = np.fft.fft(signal)
        frequencies = np.fft.fftfreq(num_samples, 1 / num_samples)
        amplitudes  = np.abs(fourier_coefficients) / num_samples

        fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
        axes[0].plot(time, signal, color='steelblue', lw=1.4)
        axes[0].set_xlabel("Time (s)")
        axes[0].set_ylabel("Amplitude")
        axes[0].grid(alpha=0.3)
        freq_limit = min(100, max(frequency_1_slider.value,
                                  frequency_2_slider.value,
                                  frequency_3_slider.value) + 20)
        axes[1].bar(frequencies, amplitudes, width=frequencies[1] - frequencies[0],
                    color='steelblue', edgecolor='none')
        axes[1].set_xlabel("Frequency (Hz)")
        axes[1].set_ylabel("Amplitude")
        axes[1].set_xlim(-freq_limit, freq_limit)
        axes[1].grid(alpha=0.3)
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    all_sliders = [amplitude_1_slider, frequency_1_slider,
                   amplitude_2_slider, frequency_2_slider,
                   amplitude_3_slider, frequency_3_slider,
                   dc_offset_slider]
    for widget in all_sliders:
        widget.observe(update_plot, names='value')
    display(VBox([HBox([amplitude_1_slider, frequency_1_slider]),
                  HBox([amplitude_2_slider, frequency_2_slider]),
                  HBox([amplitude_3_slider, frequency_3_slider]),
                  dc_offset_slider, output_area]))
    update_plot()


def wave_2d():
    """Visualise a 2-D sinusoidal wave and its power spectrum for different spatial frequencies."""
    freq_x_slider = FloatSlider(value=3.0, min=0.0, max=10.0, step=0.5,
                                description="fx (cycles/im)", style=label_style, layout=slider_layout)
    freq_y_slider = FloatSlider(value=1.0, min=0.0, max=10.0, step=0.5,
                                description="fy (cycles/im)", style=label_style, layout=slider_layout)
    phase_slider  = FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=0.25,
                                description="Phase φ",        style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        grid_size = 128
        coords = np.linspace(0, 1, grid_size)
        X, Y = np.meshgrid(coords, coords)
        wave = np.sin(2 * np.pi * freq_x_slider.value * X
                      + 2 * np.pi * freq_y_slider.value * Y
                      + phase_slider.value)
        power_spec = power_spectrum_2d(wave)

        fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
        axes[0].imshow(wave, cmap='RdBu', origin='lower', vmin=-1, vmax=1)
        axes[0].axis('off')
        axes[0].set_title(
            f"2D wave: fx={freq_x_slider.value}, fy={freq_y_slider.value}, φ={phase_slider.value:.2f}"
        )
        axes[1].imshow(power_spec, cmap='inferno', origin='lower')
        axes[1].axis('off')
        axes[1].set_title("Power spectrum")
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [freq_x_slider, freq_y_slider, phase_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([freq_x_slider, freq_y_slider, phase_slider, output_area]))
    update_plot()


def ft_patterns():
    """Static figure: show the Fourier power spectrum of several canonical image patterns."""
    grid_size = 128
    example_patterns = [
        (make_letter_image('Q', grid_size),                              "Letter Q"),
        (make_ring_pattern(grid_size),                                   "Ring"),
        (make_horizontal_line(grid_size),                                "Horiz. line"),
        (make_dot_pattern(grid_size),                                    "Small dot"),
        (np.random.default_rng(0).standard_normal((grid_size, grid_size)), "Gaussian noise"),
        (gaussian_filter(make_letter_image('Q', grid_size), sigma=5),   "Low-pass Q"),
    ]
    fig, axes = plt.subplots(2, len(example_patterns), figsize=(3.5 * len(example_patterns), 7))
    for col, (image, title) in enumerate(example_patterns):
        power_spec = power_spectrum_2d(image)
        axes[0, col].imshow(image, cmap='gray', origin='lower')
        axes[0, col].axis('off')
        axes[0, col].set_title(title, fontsize=8)
        axes[1, col].imshow(power_spec, cmap='inferno', origin='lower')
        axes[1, col].axis('off')
    axes[0, 0].set_ylabel("Real space",   fontsize=9)
    axes[1, 0].set_ylabel("Fourier space", fontsize=9)
    plt.tight_layout()
    display(figure_to_image(fig))


def ft_properties():
    """Show how translation and rotation of an image affect its Fourier amplitude and phase."""
    transform_dropdown = Dropdown(
        options=['Translation (+dx,+dy)', 'Rotation (30°)', 'Both'],
        value='Translation (+dx,+dy)',
        description="Property:", style=label_style
    )
    output_area = Output()

    def update_plot(_=None):
        chosen_transform = transform_dropdown.value
        if 'Translation' in chosen_transform:
            transformed_image = np.roll(np.roll(test_image_128px, 20, axis=0), 30, axis=1)
        elif 'Rotation' in chosen_transform:
            transformed_image = nd_rotate(test_image_128px, 30, reshape=False)
        else:
            transformed_image = nd_rotate(
                np.roll(np.roll(test_image_128px, 20, axis=0), 30, axis=1),
                30, reshape=False
            )
        transformed_fourier = np.fft.fftshift(np.fft.fft2(transformed_image))

        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        for row, (image, fourier, label) in enumerate([
            (test_image_128px,   test_image_128px_fourier, 'Original'),
            (transformed_image,  transformed_fourier,      chosen_transform),
        ]):
            axes[row, 0].imshow(image, cmap='gray', origin='lower')
            axes[row, 0].set_title(f"{label}\nReal space")
            axes[row, 0].axis('off')
            axes[row, 1].imshow(np.log(np.abs(fourier) + 1), cmap='inferno', origin='lower')
            axes[row, 1].set_title("FT amplitude")
            axes[row, 1].axis('off')
            axes[row, 2].imshow(np.angle(fourier), cmap='hsv', origin='lower', vmin=-np.pi, vmax=np.pi)
            axes[row, 2].set_title("FT phase")
            axes[row, 2].axis('off')
        amp_original    = np.log(np.abs(test_image_128px_fourier) + 1)
        amp_transformed = np.log(np.abs(transformed_fourier) + 1)
        axes[0, 3].imshow(np.abs(amp_original - amp_transformed), cmap='hot', origin='lower')
        axes[0, 3].set_title("|Δ amplitude|")
        axes[0, 3].axis('off')
        axes[1, 3].imshow(
            np.abs(np.angle(test_image_128px_fourier) - np.angle(transformed_fourier)) % np.pi,
            cmap='hot', origin='lower'
        )
        axes[1, 3].set_title("|Δ phase| mod π")
        axes[1, 3].axis('off')
        plt.suptitle(f"Fourier properties: {chosen_transform}", fontsize=10)
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    transform_dropdown.observe(update_plot, names='value')
    display(VBox([transform_dropdown, output_area]))
    update_plot()


def kernel_gallery():
    """Static figure: apply each standard kernel to the test image; display input, kernel, and output."""
    kernels = {
        'Gaussian blur (σ=2)': gaussian_kernel(9, 2.0),
        'Box blur (5×5)':      box_blur_kernel(5),
        'Laplacian':           laplacian_kernel(),
        'Sobel X':             sobel_kernel_x(),
        'Sobel Y':             sobel_kernel_y(),
        'Sharpen':             sharpen_kernel(),
    }
    fig, axes = plt.subplots(3, len(kernels), figsize=(3.5 * len(kernels), 9))
    for col, (name, kernel) in enumerate(kernels.items()):
        filtered_image = convolve2d(test_image_64px, kernel, mode='same', boundary='wrap')
        axes[0, col].imshow(test_image_64px, cmap='gray', origin='lower')
        axes[0, col].axis('off')
        axes[0, col].set_title(name, fontsize=7)
        axes[1, col].imshow(kernel, cmap='RdBu', origin='lower',
                            vmin=-abs(kernel).max(), vmax=abs(kernel).max())
        axes[1, col].axis('off')
        axes[1, col].set_title("Kernel", fontsize=7)
        axes[2, col].imshow(filtered_image, cmap='gray', origin='lower')
        axes[2, col].axis('off')
        axes[2, col].set_title("Result", fontsize=7)
    axes[0, 0].set_ylabel("Input",  fontsize=9)
    axes[1, 0].set_ylabel("Kernel", fontsize=9)
    axes[2, 0].set_ylabel("Output", fontsize=9)
    plt.tight_layout()
    display(figure_to_image(fig))


def kernel_explorer():
    """Choose a kernel and size interactively; see the convolution result and all three Fourier transforms."""
    kernel_dropdown    = Dropdown(
        options=['Gaussian blur', 'Box blur', 'Laplacian', 'Sobel X', 'Sobel Y', 'Sharpen'],
        value='Gaussian blur',
        description="Kernel:", style=label_style
    )
    kernel_size_slider = IntSlider(value=9, min=3, max=21, step=2,
                                   description="Kernel size", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        kernel_name = kernel_dropdown.value
        kernel_size = kernel_size_slider.value
        if kernel_name == 'Gaussian blur':
            kernel = gaussian_kernel(kernel_size, kernel_size / 6)
        elif kernel_name == 'Box blur':
            kernel = box_blur_kernel(kernel_size)
        elif kernel_name == 'Laplacian':
            kernel = laplacian_kernel()
        elif kernel_name == 'Sobel X':
            kernel = sobel_kernel_x()
        elif kernel_name == 'Sobel Y':
            kernel = sobel_kernel_y()
        else:
            kernel = sharpen_kernel()

        filtered_image = convolve2d(test_image_64px, kernel, mode='same', boundary='wrap')
        pad = 32 - kernel.shape[0] // 2
        padded_kernel = np.pad(kernel, (pad, pad + 1))
        power_spec_input  = power_spectrum_2d(test_image_64px)
        power_spec_kernel = power_spectrum_2d(padded_kernel)
        power_spec_output = power_spectrum_2d(filtered_image)

        fig, axes = plt.subplots(2, 3, figsize=(13, 8))
        axes[0, 0].imshow(test_image_64px, cmap='gray', origin='lower')
        axes[0, 0].axis('off')
        axes[0, 0].set_title("Input")
        axes[0, 1].imshow(kernel, cmap='RdBu', origin='lower',
                          vmin=-abs(kernel).max(), vmax=abs(kernel).max())
        axes[0, 1].axis('off')
        axes[0, 1].set_title(f"Kernel: {kernel_name}")
        axes[0, 2].imshow(filtered_image, cmap='gray', origin='lower')
        axes[0, 2].axis('off')
        axes[0, 2].set_title("Filtered")
        axes[1, 0].imshow(power_spec_input,  cmap='inferno', origin='lower')
        axes[1, 0].axis('off')
        axes[1, 0].set_title("FT input")
        axes[1, 1].imshow(power_spec_kernel, cmap='inferno', origin='lower')
        axes[1, 1].axis('off')
        axes[1, 1].set_title("FT kernel")
        axes[1, 2].imshow(power_spec_output, cmap='inferno', origin='lower')
        axes[1, 2].axis('off')
        axes[1, 2].set_title("FT output")
        plt.suptitle("Convolution theorem", fontsize=10)
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [kernel_dropdown, kernel_size_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([kernel_dropdown, kernel_size_slider, output_area]))
    update_plot()


def frequency_filter():
    """Apply a low-pass, high-pass, or band-pass filter directly in Fourier space."""
    filter_type_dropdown = Dropdown(
        options=['Low-pass', 'High-pass', 'Band-pass'],
        value='Low-pass',
        description="Filter type:", style=label_style
    )
    cutoff_low_slider  = FloatSlider(value=0.2, min=0.02, max=0.5, step=0.02,
                                     description="Cutoff f₁",      style=label_style, layout=slider_layout)
    cutoff_high_slider = FloatSlider(value=0.4, min=0.02, max=0.5, step=0.02,
                                     description="Cutoff f₂ (band)", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        image_size = test_image_64px.shape[0]
        frequency_mask = make_frequency_mask(image_size, cutoff_low_slider.value,
                                             cutoff_high_slider.value, filter_type_dropdown.value)
        fourier_image    = np.fft.fftshift(np.fft.fft2(test_image_64px))
        fourier_filtered = fourier_image * frequency_mask
        filtered_image   = np.real(np.fft.ifft2(np.fft.ifftshift(fourier_filtered)))

        fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
        axes[0].imshow(test_image_64px, cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title("Original")
        axes[1].imshow(frequency_mask, cmap='Blues', origin='lower')
        axes[1].axis('off')
        axes[1].set_title(f"Mask ({filter_type_dropdown.value})")
        axes[2].imshow(np.log(np.abs(fourier_filtered) + 1), cmap='inferno', origin='lower')
        axes[2].axis('off')
        axes[2].set_title("Filtered FT")
        axes[3].imshow(filtered_image, cmap='gray', origin='lower')
        axes[3].axis('off')
        axes[3].set_title("Result")
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [filter_type_dropdown, cutoff_low_slider, cutoff_high_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([filter_type_dropdown, cutoff_low_slider, cutoff_high_slider, output_area]))
    update_plot()


def ctf_explorer():
    """Explore how the contrast transfer function (CTF) modulates a cryo-EM image.

    Uses pixel_size = 2 Å so the Nyquist frequency (0.25 Å⁻¹) sits within the first
    few CTF oscillations — giving clearly visible Thon rings at realistic defocus values.
    """
    _images = {'Letter Q': test_image_128px}
    if tud_flame_image is not None:
        _images['TUD flame'] = tud_flame_image

    image_dropdown     = Dropdown(options=list(_images.keys()), value=list(_images.keys())[0],
                                  description="Image:",          style=label_style)
    defocus_slider     = FloatSlider(value=1.0, min=0.2,      max=5.0,      step=0.2,
                                     description="Defocus (µm)",    style=label_style, layout=slider_layout)
    voltage_slider     = FloatSlider(value=300, min=100,       max=300,      step=100,
                                     description="Voltage (kV)",    style=label_style, layout=slider_layout)
    cs_slider          = FloatSlider(value=2.7, min=0.0,       max=5.0,      step=0.1,
                                     description="Cs (mm)",         style=label_style, layout=slider_layout)
    phase_shift_slider = FloatSlider(value=0.0, min=-np.pi/2,  max=np.pi/2,  step=0.1,
                                     description="Phase shift Δφ",  style=label_style, layout=slider_layout)
    bfactor_slider     = FloatSlider(value=0.0, min=0.0,       max=200.0,    step=10.0,
                                     description="B-factor (Å²)",   style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        ref_image     = _images[image_dropdown.value]
        wavelength_A  = relativistic_electron_wavelength(voltage_slider.value * 1e3)
        pixel_size_A  = 2.0                        # Å per pixel
        nyquist       = 0.5 / pixel_size_A         # Å⁻¹
        spatial_freqs = np.linspace(0, nyquist, 1024)
        defocus_A     = defocus_slider.value * 1e4  # µm → Å
        cs_A          = cs_slider.value * 1e7       # mm → Å

        gamma_1d  = ((-np.pi / 2) * cs_A * wavelength_A ** 3 * spatial_freqs ** 4
                     + np.pi * wavelength_A * defocus_A * spatial_freqs ** 2)
        ctf_curve = -np.sin(phase_shift_slider.value + gamma_1d)
        if bfactor_slider.value > 0:
            ctf_curve *= np.exp(-bfactor_slider.value * spatial_freqs ** 2)

        image_size = ref_image.shape[0]
        ky = np.fft.fftfreq(image_size, pixel_size_A)
        kx = np.fft.rfftfreq(image_size, pixel_size_A)
        KX, KY = np.meshgrid(kx, ky)
        radial_freq_2d = np.sqrt(KX ** 2 + KY ** 2)   # Å⁻¹
        gamma_2d  = ((-np.pi / 2) * cs_A * wavelength_A ** 3 * radial_freq_2d ** 4
                     + np.pi * wavelength_A * defocus_A * radial_freq_2d ** 2)
        ctf_2d    = -np.sin(phase_shift_slider.value + gamma_2d)
        if bfactor_slider.value > 0:
            ctf_2d *= np.exp(-bfactor_slider.value * radial_freq_2d ** 2)

        image_ctf_modulated = np.fft.irfftn(np.fft.rfftn(ref_image) * ctf_2d, ref_image.shape)
        image_noisy = image_ctf_modulated + np.random.default_rng(1).standard_normal(ref_image.shape) * 0.1

        # Full 2D frequency grid (not the rfft half-plane) so the centred power spectrum is symmetric
        freq_axis = np.fft.fftshift(np.fft.fftfreq(image_size, pixel_size_A))
        KXf, KYf = np.meshgrid(freq_axis, freq_axis)
        radial_freq_full = np.sqrt(KXf ** 2 + KYf ** 2)   # Å⁻¹
        gamma_full = ((-np.pi / 2) * cs_A * wavelength_A ** 3 * radial_freq_full ** 4
                      + np.pi * wavelength_A * defocus_A * radial_freq_full ** 2)
        ctf_full = -np.sin(phase_shift_slider.value + gamma_full)
        if bfactor_slider.value > 0:
            ctf_full *= np.exp(-bfactor_slider.value * radial_freq_full ** 2)
        ctf_power = ctf_full ** 2

        fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
        axes[0].plot(spatial_freqs, ctf_curve, color='steelblue', lw=1.8)
        axes[0].axhline(0, color='k', lw=0.8, ls='--')
        axes[0].set_xlabel("Spatial freq (Å⁻¹)")
        axes[0].set_ylabel("CTF")
        axes[0].set_title(f"CTF (Δf={defocus_slider.value} µm, V={voltage_slider.value:.0f} kV)")
        axes[0].set_ylim(-1.2, 1.2)
        axes[0].set_xlim(0, nyquist)
        axes[0].grid(alpha=0.3)
        axes[1].imshow(ref_image, cmap='gray', origin='lower')
        axes[1].axis('off')
        axes[1].set_title("True image")
        axes[2].imshow(image_noisy, cmap='gray', origin='lower',
                       vmin=ref_image.min(), vmax=ref_image.max())
        axes[2].axis('off')
        axes[2].set_title(f"CTF-modulated (B={bfactor_slider.value:.0f} Å²)")
        axes[3].imshow(ctf_power, cmap='inferno', origin='lower', vmin=0, vmax=1,
                       extent=[-nyquist, nyquist, -nyquist, nyquist])
        axes[3].set_xlabel("$k_x$ (Å⁻¹)")
        axes[3].set_ylabel("$k_y$ (Å⁻¹)")
        axes[3].set_title("CTF\u00b2 (Thon ring pattern)")
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [image_dropdown, defocus_slider, voltage_slider, cs_slider,
                   phase_shift_slider, bfactor_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([image_dropdown, defocus_slider, voltage_slider, cs_slider,
                  phase_shift_slider, bfactor_slider, output_area]))
    update_plot()


def ctf_correction():
    """Compare three strategies for correcting CTF-induced phase reversals in Fourier space."""
    _images = {'Letter Q': test_image_128px}
    if tud_flame_image is not None:
        _images['TUD flame'] = tud_flame_image

    image_dropdown      = Dropdown(options=list(_images.keys()), value=list(_images.keys())[0],
                                   description="Image:",         style=label_style)
    correction_dropdown = Dropdown(
        options=['Phase flip', 'Divide by CTF', 'Wiener filter'],
        value='Wiener filter',
        description="Correction:", style=label_style
    )
    snr_slider       = FloatSlider(value=1.0,  min=0.01, max=10.0, step=0.1,
                                   description="SNR (Wiener)",  style=label_style, layout=slider_layout)
    threshold_slider = FloatSlider(value=0.1,  min=0.01, max=0.5,  step=0.01,
                                   description="Threshold ε",   style=label_style, layout=slider_layout)
    defocus_slider   = FloatSlider(value=1.0,  min=0.2,  max=5.0,  step=0.2,
                                   description="Defocus (µm)",  style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        ref_image    = _images[image_dropdown.value]
        wavelength_A = relativistic_electron_wavelength(300e3)
        pixel_size_A = 2.0
        image_size   = ref_image.shape[0]
        ky = np.fft.fftfreq(image_size, pixel_size_A)
        kx = np.fft.rfftfreq(image_size, pixel_size_A)
        KX, KY = np.meshgrid(kx, ky)
        radial_freq_2d = np.sqrt(KX ** 2 + KY ** 2)   # Å⁻¹
        defocus_A = defocus_slider.value * 1e4
        cs_A      = 2.7e7
        ctf_2d = -np.sin((-np.pi / 2) * cs_A * wavelength_A ** 3 * radial_freq_2d ** 4
                         + np.pi * wavelength_A * defocus_A * radial_freq_2d ** 2)

        fourier_true     = np.fft.rfftn(ref_image)
        noise_fourier    = np.random.default_rng(1).standard_normal(fourier_true.shape) * 0.3
        fourier_observed = fourier_true * ctf_2d + noise_fourier

        method = correction_dropdown.value
        if method == 'Phase flip':
            # Multiply by sign(CTF) — restores phase but not amplitude
            fourier_corrected = fourier_observed * np.sign(ctf_2d)
        elif method == 'Divide by CTF':
            # Direct division; suppress frequencies where |CTF| < ε to avoid dividing by ~0
            fourier_corrected = np.where(
                np.abs(ctf_2d) >= threshold_slider.value,
                fourier_observed / ctf_2d,
                0.0
            )
        else:
            # Wiener filter: regularised inversion — high SNR → divide aggressively; low SNR → attenuate
            fourier_corrected = fourier_observed * ctf_2d / (ctf_2d ** 2 + 1.0 / snr_slider.value)

        image_corrected     = np.fft.irfftn(fourier_corrected, ref_image.shape)
        image_ctf_modulated = np.fft.irfftn(fourier_observed,  ref_image.shape)
        rmse = np.sqrt(np.mean((image_corrected - ref_image) ** 2))

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(ref_image, cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title("True image")
        axes[1].imshow(image_ctf_modulated, cmap='gray', origin='lower')
        axes[1].axis('off')
        axes[1].set_title(f"CTF-modulated (Δf={defocus_slider.value} µm)")
        axes[2].imshow(image_corrected, cmap='gray', origin='lower')
        axes[2].axis('off')
        axes[2].set_title(f"Corrected: {method}\nRMSE = {rmse:.3f}")
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [image_dropdown, correction_dropdown, snr_slider, threshold_slider, defocus_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([image_dropdown, defocus_slider, correction_dropdown,
                  threshold_slider, snr_slider, output_area]))
    update_plot()


def delocalization_explorer():
    """Show how CTF-induced delocalization smears signal away from its true position.

    The delocalization length estimate d ≈ sqrt(λ · Δf) gives the characteristic
    radius over which a point-like feature spreads after CTF application.
    """
    _images = {'Letter Q': test_image_128px}
    if tud_flame_image is not None:
        _images['TUD flame'] = tud_flame_image

    image_dropdown = Dropdown(options=list(_images.keys()), value=list(_images.keys())[0],
                               description="Image:",       style=label_style)
    defocus_slider = FloatSlider(value=1.0, min=0.2, max=5.0, step=0.2,
                                  description="Defocus (µm)", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        pixel_size_A = 2.0
        wavelength_A = relativistic_electron_wavelength(300e3)
        defocus_A    = defocus_slider.value * 1e4
        cs_A         = 2.7e7

        image = _images[image_dropdown.value]
        n = image.shape[0]

        ky = np.fft.fftfreq(n, pixel_size_A)
        kx = np.fft.rfftfreq(n, pixel_size_A)
        KX, KY = np.meshgrid(kx, ky)
        radial_freq_2d = np.sqrt(KX ** 2 + KY ** 2)
        gamma_2d = ((-np.pi / 2) * cs_A * wavelength_A ** 3 * radial_freq_2d ** 4
                    + np.pi * wavelength_A * defocus_A * radial_freq_2d ** 2)
        ctf_2d = -np.sin(gamma_2d)

        image_ctf = np.real(np.fft.irfftn(np.fft.rfftn(image) * ctf_2d, image.shape))

        # CTF PSF: inverse Fourier transform of the CTF (the real-space spread function)
        ctf_psf  = np.fft.fftshift(np.real(np.fft.irfft2(ctf_2d, s=(n, n))))
        crop     = min(64, n // 2)
        c        = n // 2
        psf_crop = ctf_psf[c - crop : c + crop, c - crop : c + crop]

        delocal_A  = np.sqrt(wavelength_A * defocus_A)
        delocal_px = delocal_A / pixel_size_A

        fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
        axes[0].imshow(image, cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title("True object")
        axes[1].imshow(image_ctf, cmap='gray', origin='lower',
                       vmin=image.min(), vmax=image.max())
        axes[1].axis('off')
        axes[1].set_title(f"After CTF (Δf = {defocus_slider.value} µm)")
        axes[2].imshow(np.abs(image - image_ctf), cmap='hot', origin='lower')
        axes[2].axis('off')
        axes[2].set_title(f"|Delocalized signal|\nd ≈ {delocal_A:.0f} Å  ({delocal_px:.1f} px)")
        vmax = np.abs(psf_crop).max()
        axes[3].imshow(psf_crop, cmap='RdBu', origin='lower', vmin=-vmax, vmax=vmax)
        axes[3].axis('off')
        axes[3].set_title("CTF point spread function\n(real-space spread)")
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [image_dropdown, defocus_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([image_dropdown, defocus_slider, output_area]))
    update_plot()


print("Setup complete.")

# Practical: Fourier Optics and Image Processing

This practical covers the Fourier transform and its applications in image processing and electron microscopy.

> **To start the practicals:** Either run the code cells one by one or click **Run All** (⏭) in the toolbar above.

***

## Getting familiar with waves

Let us first consider signals involving a single independent variable, and for convenience let us refer to this independent variable as time $t$ (while really, it could be anything).

A sinusoidal wave has three fundamental parameters:

$$
f(t) = A \cdot \cos(2\pi f \cdot t - \varphi)
$$

| Parameter | Symbol | Effect |
|-----------|--------|--------|
| Amplitude | $A$    | Height of the wave |
| Frequency | $f$    | Number of cycles per unit time |
| Phase     | $\varphi$ | Horizontal shift |

In [ ]:
wave_explorer()

Processing arbitray signals requires a mathematical way to describe these signals. For real life data, a mathematical description would require near infinite amounts of parameters to describe the signal and this mathematical model will be different for every signal. Fortunately, as we shall see in this course, if the signal is periodic, or can be made periodic, then __all__ such signals can be expressed as a sum of simple sinusoids.

In the following, we will focus on how we can employ this concept for the analysis of images, but more generally you will obtain an intuitive understanding of the toolbox of Fourier theory that can be applicable to a wide variety of scientific propblems.


***

## Fourier series

In this section we shall see that _any_ 1-dimensional signal (even a discontinuous one) can be expressed as a sum of simple sine and cosine waves. This counter-intuitive idea was first developed by Joseph Fourier (1768–1830) while studying heat conduction.

Fourier showed that the heat equation, a partial differential equation governing how temperature evolves in a conductor, is easily solved when the spatial distribution is a single sinusoid. For more realistic boundary conditions he argued that _any_ temperature distribution could be written as an infinite sum of sinusoids, a decomposition now called a **Fourier Series**. This insight gave rise to the broader framework of __Fourier Analysis__.

### Fourier series and the box function
    
To begin understanding this analysis technique, let us now consider a type of discontinuous function called the box function. A box function $b(x)$ has the following property: 
    

<a id='equation_2'></a>
$$
b(x) = \begin{cases} 1 & -a/2 < x < a/2 \\ 0 & \text{elsewhere} \end{cases}
$$

As we learnt [before](hri_practical01_part1#equation_1), a sinusoidal wave requires three parameters: Amplitude: $A$, Frequency: $f$, Phase: $\phi$.  

Thus, if we were to break up this functions into several sinusoidal waves, we need three parameters for each of these waves. They can be found using the Fourier Series. 

Any periodic signal can be expressed as a sum of sinusoidal waves (**Fourier series**):

$$
b(x) = A_0 + \sum_{k=1}^{F} A_k \cdot \cos\!\left(\frac{2\pi k x}{P} - \varphi_k\right)
$$

where $F$ is the maximum frequency, $P$ the period, and $A_k$, $\varphi_k$ the amplitude and phase at frequency $k$.

The amplitudes and phases are computed from the signal via the **Fourier coefficients**:

$$
a_k = \frac{2}{P}\sum_{i=0}^{P} b(i)\cos\!\frac{2\pi k i}{P}, \qquad
b_k = \frac{2}{P}\sum_{i=0}^{P} b(i)\sin\!\frac{2\pi k i}{P}, \qquad
A_k = \sqrt{a_k^2 + b_k^2}
$$

### The box function

A classic example is the box (rectangular) function:

$$
b(x) = \begin{cases} 1 & -a/2 < x < a/2 \\ 0 & \text{elsewhere} \end{cases}
$$

Its Fourier series contains only odd harmonics, and the coefficients decay as $1/k$. As we add more terms, the reconstruction improves but _Gibbs ringing_ remains at the edges. Sharp edges require infinitely many frequency components to represent perfectly. When we keep only a finite number, the approximation creates ripples near the edge.

In [ ]:
fourier_series()

***

## Part 3 – Frequency spectrum (1D FFT)

Instead of computing Fourier coefficients term by term, the **discrete Fourier transform (DFT)** computes all of them at once:

$$
F(k) = \frac{1}{P} \sum_{m=0}^{P-1} b(m) \cdot e^{-i 2\pi k m / P}
$$

The fast Fourier transform (FFT) computes this in $O(N \log N)$ rather than $O(N^2)$. The **frequency spectrum** (plot of $|F(k)|$ vs $k$) reveals which frequencies are present in the signal.

### Nyquist sampling theorem

If a signal contains no frequencies higher than $W$ Hz, it is fully determined by samples taken every $1/(2W)$ seconds:

$$
f_\text{max} \leq \frac{1}{2 \Delta t} \quad \Leftrightarrow \quad \Delta t \leq \frac{1}{2 f_\text{max}}
$$

The **Nyquist frequency** $f_N = 1/(2\Delta t)$ is the highest frequency measurable at a given sampling rate. In EM images, the pixel size $\Delta x$ sets $f_N = 1/(2\Delta x)$, and only features larger than $2\Delta x$ can be resolved.

In [ ]:
spectrum_explorer()

```{admonition} Question 1
:class: seealso
Given a detector that records 15 frames per 100 s. What is the maximum frequency component that can be recovered? What does this imply for the pixel size required to resolve a feature of width 2 Å in a cryo-EM image?
```

---

## Part 4 – 2D Fourier analysis

Images are 2D signals. A 2D sinusoidal wave has the form:

$$
f(x,y) = A \cdot \sin(2\pi f_x x + 2\pi f_y y + \varphi)
$$

with separate spatial frequencies $f_x$ (cycles per pixel along $x$) and $f_y$ (cycles per pixel along $y$). The 2D Fourier transform of an image gives the amplitude and phase at every spatial frequency $(f_x, f_y)$.

### Properties

**Translation property**: Shifting an image in real space does not change the amplitude spectrum — only the phase spectrum.

**Rotation property**: Rotating an image rotates its Fourier transform by the same angle.

The interactive below shows 2D sine waves at different spatial frequencies, and the effects of translation and rotation on the Fourier transform of a real image.

In [ ]:
wave_2d()

### 2D FT of real images

Different image features produce characteristic Fourier signatures. A ring in real space produces a ring in Fourier space; a line produces a perpendicular line through the origin.

In [ ]:
ft_patterns()

### Translation and rotation properties

In [ ]:
ft_properties()

```{admonition} Question 2
:class: seealso
Play with the translation and rotation of the image and observe the amplitude spectrum. What changes and what stays the same in each case? Can you explain why?
```

---

## Part 5 – The convolution theorem

**Convolution** describes how one function modifies another by sweeping over all positions:

$$
(f * g)(y) = \int_{-\infty}^{\infty} f(x)\,g(y-x)\,dx
$$

In image processing, convolving an image with a **kernel** (a small matrix) applies a local operation: smoothing, sharpening, or edge detection. The **convolution theorem** states:

$$
\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}
$$

Convolution in real space equals *multiplication* in Fourier space. This makes filtering in Fourier space much faster for large kernels than direct spatial convolution.

### Common kernels

In [ ]:
kernel_gallery()

### Interactive kernel

In [ ]:
kernel_explorer()

```{admonition} Question 3
:class: seealso
What does the FT of a Gaussian blur kernel look like? What does this tell you about which spatial frequencies are attenuated? How does this compare to the Laplacian kernel?
```

---

## Part 6 – Low-pass and high-pass filtering in Fourier space

We can design filters by directly modifying the Fourier transform: set Fourier coefficients outside (or inside) a radius to zero before inverse-transforming. This is equivalent to convolving with the Fourier transform of the mask function.

In [ ]:
frequency_filter()

***

## Part 7 – The contrast transfer function (CTF)

In phase-contrast electron microscopy, the image intensity is related to the projected potential of the specimen through the **contrast transfer function (CTF)**. The CTF modulates the Fourier amplitudes as a function of spatial frequency $k$:

$$
\text{CTF}(k) = -\sin\!\left[\Delta\varphi + \frac{-\pi}{2}C_s\lambda^3 k^4 + \pi\lambda\Delta_f k^2\right]
$$

where:

| Symbol | Quantity |
|--------|----------|
| $k$ | spatial frequency (Å⁻¹) |
| $C_s$ | spherical aberration coefficient |
| $\lambda$ | relativistic electron wavelength |
| $\Delta_f$ | defocus (positive = underfocus) |
| $\Delta\varphi$ | additional phase shift (e.g. from phase plate) |

The electron wavelength depends on the accelerating voltage $V$ (in eV):

$$
\lambda = \frac{12.264}{\sqrt{V(1 + V \cdot 0.98 \times 10^{-6})}} \; \text{Å}
$$

Defocusing causes **phase reversals** at specific spatial frequencies (the CTF zeros). Frequencies near these zeros are not faithfully transferred to the image.

In [ ]:
ctf_explorer()

### CTF correction

To recover the true image from a CTF-modulated observation, we need to invert the CTF. Three standard approaches are compared below.

**Method 1 – Phase flip**: Multiply every Fourier coefficient by the sign of the CTF:

$$
\hat{F}(k) = F_\text{obs}(k) \cdot \text{sign}[\text{CTF}(k)]
$$

This restores the correct phase at every spatial frequency, but does not correct the amplitude — frequencies near CTF zeros (where the CTF is small) remain poorly recovered.

**Method 2 – Divide by CTF**: Direct inversion of the CTF:

$$
\hat{F}(k) = \frac{F_\text{obs}(k)}{\text{CTF}(k)}
$$

Dividing by a value close to zero amplifies noise enormously near CTF zeros. The threshold $\varepsilon$ guards against this: frequencies where $|\text{CTF}(k)| < \varepsilon$ are set to zero instead of divided. A large $\varepsilon$ discards more frequencies but avoids noise blow-up; a small $\varepsilon$ keeps more information but may amplify noise.

**Method 3 – Wiener filter**: Regularised inversion that jointly optimises resolution and noise suppression:

$$
\hat{F}(k) = \frac{F_\text{obs}(k) \cdot \text{CTF}(k)}{\text{CTF}(k)^2 + \text{SNR}^{-1}}
$$

The SNR (signal-to-noise ratio) is a regularisation parameter. When SNR is large (clean data), the denominator ≈ CTF² and the filter approaches full CTF inversion. When SNR is small (noisy data), the denominator is dominated by SNR⁻¹ and the filter attenuates rather than amplifies — never dividing by a small number. Adjust the SNR slider to match the noise level of the simulated image.

In [ ]:
ctf_correction()

### Signal delocalization

A direct consequence of the CTF is **signal delocalization**: because the CTF is not flat, it acts as a spatially extended filter. In real space, the CTF corresponds to a point spread function (PSF) that spreads signal from each image point over a surrounding area. This means that even a single atom or sharp edge is not recorded at its true position — its contrast is redistributed over a radius set by the CTF.

A rough estimate of the delocalization length is:

$$
d \approx \sqrt{\lambda \cdot \Delta_f}
$$

where $\lambda$ is the electron wavelength and $\Delta_f$ is the defocus. At $\Delta_f = 1\,\mu\text{m}$ and 300 kV, $d \approx 14\,\text{Å}$. Larger defocus = more delocalization = sharper Thon rings but blurrier features.

The interactive below uses the TUD flame image to show how increasing defocus delocalizes signal away from its true origin, and what the corresponding PSF looks like in real space.

In [ ]:
delocalization_explorer()

```{admonition} Question 4
:class: seealso
Compare the three CTF correction methods. What are the trade-offs?
- Phase flipping: what information is still lost?
- Full CTF correction with threshold: what happens to frequencies near the CTF zeros?
- Wiener filter: what is the optimal SNR for your simulated data?
```

---

## Summary

| Topic | Key concept |
|-------|-------------|
| Sinusoidal waves | $f(t) = A\cos(2\pi f t - \varphi)$; three parameters: $A$, $f$, $\varphi$ |
| Fourier series | Any periodic signal = sum of sinusoids; coefficients via $a_k$, $b_k$ integrals |
| FFT | Fast algorithm ($O(N\log N)$) for discrete FT; frequency spectrum = $|F(k)|$ |
| Nyquist | Sampling at $\Delta t$ can recover frequencies up to $1/(2\Delta t)$ |
| 2D FT | Generalises 1D; amplitude spectrum = $|F(f_x,f_y)|$; rotation and translation properties |
| Convolution theorem | $\mathcal{F}\{f*g\} = \mathcal{F}\{f\}\cdot\mathcal{F}\{g\}$; filter in Fourier space |
| CTF | $-\sin(\gamma(k))$; phase reversals at zeros; correct by phase flip, division, or Wiener filter |